# 🈴 Kanji SensAi — An Interactive Japanese Kanji Tutor

## Overview

**Kanji SensAi** is an AI-powered Japanese language assistant that specializes in teaching kanji characters through a multimodal experience. When a user asks about a word, the agent delivers a full lesson that includes:

- **Text explanation** — the kanji character, its meaning, origin, readings (on'yomi / kun'yomi), romaji, hiragana, and a mnemonic tip.
- **Audio narration** — a natural-sounding voice lesson with correct Japanese pronunciation.
- **Visual mnemonic** — a generated image showing a 4-panel transition from a concept illustration to the kanji in brush calligraphy style.

Beyond kanji lookups, the agent can also chat naturally about Japanese culture, grammar, history, anime, food, and more.

## Architecture

The project is built around a layered, protocol-based design using Python dataclasses:

| Layer | Class | Responsibility |
|-------|-------|---------------|
| **LLM Clients** | `LlmChat`, `LlmVoice`, `LlmImage` | Wrap OpenAI APIs for text, speech, and image generation |
| **Agent** | `KanjiSenseiAgent` | Orchestrates intent detection (via tool calling) and the full kanji lookup pipeline |
| **UI** | `KanjiSenseiUI` | Gradio interface with chat, audio player, image display, and loading states |

## Key Features

- **Tool calling** — Uses OpenAI function calling to detect when the user wants a kanji lesson vs. general conversation.
- **Multimodal output** — Combines `gpt-4.1-mini` (chat), `gpt-4o-mini-tts` (voice), and `gpt-image-1-mini` (image) in a single pipeline.
- **Loading UX** — Shows a waiting message and disables input while the lesson is being generated, then reveals the full result.
- **Greeting on launch** — The assistant introduces itself in Japanese and English when the app starts.

In [ ]:
import json
from dataclasses import dataclass
from typing_extensions import Protocol
import gradio as gr

from openai import OpenAI

In [ ]:
@dataclass(frozen=True, slots=True)
class LlmClient(Protocol):
    client: OpenAI
    model_name: str
    system_message: str

In [ ]:
@dataclass(frozen=True, slots=True)
class LlmChat(LlmClient):
    tools: list[dict] | None = None

    def _build_messages(
        self,
        *,
        messages: list[dict[str, str]],
        history: list[dict[str, str]] | None = None,
    ) -> list[dict[str, str]]:
        hist = [
            {"role": h["role"], "content": h["content"]} for h in history
        ] if history else []
        return [{"role": "system", "content": self.system_message}] + hist + messages

    def chat(self, message: str, history: list[dict[str, str]]) -> str:
        return self.client.chat.completions.create(
            model=self.model_name,
            messages=self._build_messages(
                messages=[{"role": "user", "content": message}],
                history=history,
            ),
        ).choices[0].message.content

    def complete(self, message: str, history: list[dict[str, str]]):
        kwargs = dict(
            model=self.model_name,
            messages=self._build_messages(
                messages=[{"role": "user", "content": message}],
                history=history,
            ),
        )
        if self.tools:
            kwargs["tools"] = self.tools
        return self.client.chat.completions.create(**kwargs).choices[0].message

In [ ]:
import tempfile

@dataclass(frozen=True, slots=True)
class LlmVoice(LlmClient):
    def talk(self, message: str, voice: str = "onyx") -> str:
        response = self.client.audio.speech.create(
            model=self.model_name,
            voice=voice,
            input=message,
            instructions=self.system_message,
        )
        tmp = tempfile.NamedTemporaryFile(suffix=".mp3", delete=False)
        response.stream_to_file(tmp.name)
        return tmp.name

In [ ]:
import base64
import requests as http_requests

@dataclass(frozen=True, slots=True)
class LlmImage(LlmClient):
    def _save_temporal_image(self, *, image_url: str | None = None, b64_data: str | None = None) -> str:
        tmp = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
        if b64_data:
            tmp.write(base64.b64decode(b64_data))
        elif image_url:
            tmp.write(http_requests.get(image_url).content)
        tmp.flush()
        return tmp.name

    def draw(self, prompt: str) -> str:
        full_prompt = f"{self.system_message}\n\n{prompt}" if self.system_message else prompt
        response = self.client.images.generate(
            model=self.model_name,
            prompt=full_prompt,
            size="1024x1024",
        )
        item = response.data[0]
        return self._save_temporal_image(image_url=item.url, b64_data=item.b64_json)

In [ ]:
@dataclass(frozen=True, slots=True)
class LlmAgent(Protocol):
    chat_client: LlmChat
    voice_client: LlmVoice
    image_client: LlmImage

In [ ]:
@dataclass(frozen=True, slots=True)
class KanjiSenseiAgent(LlmAgent):

    def detect(self, message: str, history: list[dict[str, str]] | None = None) -> tuple[bool, str]:
        history = history or []
        response = self.chat_client.complete(message=message, history=history)
        if response.tool_calls:
            word = json.loads(response.tool_calls[0].function.arguments)["word"]
            return True, word
        return False, response.content

    def lookup(self, word: str, history: list[dict[str, str]] | None = None) -> dict:
        history = history or []
        explanation = self.chat_client.chat(
            message=(
                f"The user wants to learn the kanji for: {word}\n\n"
                "Provide the full kanji explanation now. Do NOT call any tool. "
                "Just respond directly with the kanji lesson."
            ),
            history=history,
        )
        audio_path = self.voice_client.talk(message=explanation)
        image_path = self.image_client.draw(prompt=explanation)
        return {
            "explanation": explanation,
            "audio_path": audio_path,
            "image_path": image_path,
        }

In [ ]:
KANJI_LOOKUP_TOOL = {
    "type": "function",
    "function": {
        "name": "lookup_kanji",
        "description": (
            "Look up detailed kanji information for a word, generating a full lesson "
            "with visual mnemonic and audio narration."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "word": {
                    "type": "string",
                    "description": "The word to look up (in any language)",
                }
            },
            "required": ["word"],
            "additionalProperties": False,
        },
    },
}

sensei = KanjiSenseiAgent(
    chat_client=LlmChat(
        client=OpenAI(),
        model_name="gpt-4.1-mini",
        tools=[KANJI_LOOKUP_TOOL],
        system_message=(
            "You are Kanji Sensei, a friendly and knowledgeable Japanese language teacher. "
            "You can chat naturally about anything related to Japan and the Japanese language — "
            "culture, grammar, vocabulary, history, travel tips, anime, food, etc.\n\n"
            "If a question is clearly unrelated to Japan or the Japanese language, politely decline "
            "and steer the conversation back to your area of expertise.\n\n"
            "When the user wants to learn about a specific word's kanji (e.g. they ask "
            "'how do you write mountain in kanji?', or simply send a single word like 'fire'), "
            "use the `lookup_kanji` tool to trigger a full kanji lesson.\n"
            "Only call the tool when the user is clearly asking to learn a word or its kanji. "
            "For general conversation, just respond normally without tools.\n\n"
            "When explaining a kanji (after a lookup), always include:\n"
            "- The kanji character itself (e.g. 山)\n"
            "- Its meaning and origin\n"
            "- The romaji transcription (e.g. 'yama')\n"
            "- The hiragana transcription (e.g. やま)\n"
            "- The on'yomi (Chinese reading) and kun'yomi (Japanese reading)\n"
            "- A brief mnemonic tip to remember it\n\n"
            "If no kanji exists for a requested word, say so clearly and suggest "
            "3 alternative words that DO have well-known kanji. "
            "Keep explanations concise but informative. Use markdown formatting."
        ),
    ),
    voice_client=LlmVoice(
        client=OpenAI(),
        model_name="gpt-4o-mini-tts",
        system_message=(
            "You are a warm and patient Japanese language teacher narrating a kanji lesson. "
            "Speak clearly and at a calm pace. When you encounter Japanese words, "
            "pronounce them with correct Japanese pronunciation. "
            "Pause briefly between the different sections of the explanation "
            "(meaning, readings, mnemonic) so the student can follow along."
        ),
    ),
    image_client=LlmImage(
        client=OpenAI(),
        model_name="gpt-image-1-mini",
        system_message=(
            "You are generating a visual mnemonic learning aid for a Japanese kanji character. "
            "Create a sequence of 4 panels arranged left-to-right showing a smooth visual transition:\n"
            "- Panel 1: A simple, recognizable illustration of the concept the kanji represents "
            "(e.g. a mountain for 山, a tree for 木, fire for 火) in a clean, elegant style\n"
            "- Panel 2: The illustration starts morphing — its outline subtly begins to resemble the kanji strokes\n"
            "- Panel 3: The shape is mostly the kanji now, but with faint traces of the original image still visible\n"
            "- Panel 4: The clean, bold kanji character rendered in black brush calligraphy (shodō) style\n\n"
            "Use a subtle Japanese-inspired background throughout all panels — "
            "soft watercolor cherry blossoms, gentle ink wash texture, or traditional washi paper feel. "
            "Use a warm, muted color palette (cream, light pink, pale grey, hint of red). "
            "The style should feel like an elegant Japanese textbook illustration. "
            "Do NOT include any text labels or annotations — only the visual progression."
        ),
    ),
)

In [ ]:
GREETING_MESSAGE = (
    "こんにちは! (Konnichiwa!) 👋\n\n"
    "I'm **Kanji SensAi**, your Japanese language and culture companion! "
    "I'm here to help you dive into the fascinating world of Japan — from history and traditions "
    "to grammar, vocabulary, and everyday life.\n\n"
    "I'm especially a **kanji specialist**: whenever you ask me about a word, "
    "I'll give you a full lesson complete with 🎧 **audio narration** and 🖼️ **visual mnemonic**!\n\n"
    "Type any word or ask me anything about Japan to get started!"
)


@dataclass(frozen=True, slots=True)
class KanjiSenseiUI:
    agent: KanjiSenseiAgent

    def _respond(self, message: str, history: list[dict[str, str]]):
        history = history + [{"role": "user", "content": message}]

        is_lookup, result = self.agent.detect(message=message, history=history[:-1])

        if is_lookup:
            # 1. Notify user before heavy processing — disable input + button
            wait_msg = (
                f"🔄 Looking up the kanji for **{result}** — "
                "generating explanation, audio lesson, and visual mnemonic. One moment…"
            )
            yield (
                history + [{"role": "assistant", "content": wait_msg}],
                gr.update(),
                gr.update(),
                gr.update(interactive=False),
                gr.update(interactive=False),
            )

            # 2. Full pipeline: explanation + voice + image
            lesson = self.agent.lookup(word=result, history=history[:-1])

            # 3. Show explanation, deliver audio & image, re-enable controls
            yield (
                history + [
                    {"role": "assistant", "content": wait_msg},
                    {"role": "assistant", "content": lesson["explanation"]},
                ],
                lesson["audio_path"],
                lesson["image_path"],
                gr.update(value="", interactive=True),
                gr.update(interactive=True),
            )
        else:
            # Normal conversation — keep existing media untouched
            yield (
                history + [{"role": "assistant", "content": result}],
                gr.update(),
                gr.update(),
                gr.update(value=""),
                gr.update(),
            )

    def launch(self, inbrowser: bool = True):
        with gr.Blocks(title="🈴 Kanji SensAi") as demo:
            gr.Markdown(
                "# 🈴 Kanji SensAi\n"
                "Chat about Japan & Japanese, or type a word to get a full kanji lesson "
                "with audio and a visual mnemonic!"
            )

            with gr.Row():
                with gr.Column(scale=3):
                    chatbot = gr.Chatbot(
                        type="messages",
                        label="Conversation",
                        height=500,
                        value=[{"role": "assistant", "content": GREETING_MESSAGE}],
                    )
                with gr.Column(scale=1):
                    image = gr.Image(label="Kanji Mnemonic", height=300)
                    audio = gr.Audio(label="Lesson Audio", autoplay=True)

            with gr.Row():
                word_input = gr.Textbox(
                    placeholder="Ask about Japanese, or type a word to learn its kanji…",
                    show_label=False,
                    scale=5,
                )
                submit_btn = gr.Button("Ask SensAi 🎓", variant="primary", scale=1)

            outputs = [chatbot, audio, image, word_input, submit_btn]

            submit_btn.click(
                fn=self._respond,
                inputs=[word_input, chatbot],
                outputs=outputs,
            )
            word_input.submit(
                fn=self._respond,
                inputs=[word_input, chatbot],
                outputs=outputs,
            )

        demo.launch(inbrowser=inbrowser)

In [ ]:
ui = KanjiSenseiUI(agent=sensei)
ui.launch()